# Bài 7 · Xử lý dữ liệu chuỗi

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Dùng họ `.str` để chuẩn hoá, lọc, sửa, tách cột văn bản (nhớ bẫy NaN).
2. Đọc–viết được regex lõi (`\d \w \s`, `+ * ?`, `( )`, `|`) và dùng `str.extract`.
3. Biến cột chữ thành **cột tín hiệu** phân tích được — và biết quy trình kiểm chứng một regex.

In [ ]:
import json
import pandas as pd

URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")
df = pd.read_csv(URL_FULL, usecols=["id", "name", "amenities", "price",
                                    "neighbourhood_cleansed"])
df["price_num"] = (df["price"].str.replace("$", "", regex=False)
                              .str.replace(",", "", regex=False)
                              .astype(float))
print(df.shape)
df[["name", "price"]].head(3)

## 1. Họ `.str` — thao tác chuỗi chạy cả cột

In [ ]:
# Chuẩn hoá: cùng một quận, ba kiểu viết
s = pd.Series(["  Ñuñoa ", "ñuñoa", "NUNOA?  "])
print(s.tolist(), "->", s.str.strip().str.lower().tolist())

In [ ]:
# contains: bao nhiêu tên phòng khoe "gần metro"?
gan_metro = df["name"].str.contains("metro", case=False, na=False)
print(f"{gan_metro.sum()} tên ({gan_metro.mean():.0%})")

In [ ]:
# Bẫy NaN: cột chuỗi có ô trống -> contains trả None, lọc là "dính"
s2 = pd.Series(["Depto centro", None, "Casa vista"])
print("Không na=False:", s2.str.contains("centro").tolist())
print("Có    na=False:", s2.str.contains("centro", na=False).tolist())

In [ ]:
# split + explode: cột amenities là danh sách JSON trong MỘT ô
print(df["amenities"].iloc[0][:90], "…")

tien_nghi = df["amenities"].map(json.loads).explode()
tien_nghi.value_counts().head(10)

Câu hỏi phụ đáng giá: tiện nghi nào *hiếm* mà nghe sang? Thử `value_counts().tail(20)` —
"đãi cát tìm vàng" kiểu này là đặc sản của cột đa trị.

## 2. Regex — mô tả hình dáng

Bài toán mẫu: moi **số phòng ngủ** từ tên phòng ("Modern 1BR Oasis", "Casa 3 dorm centro"…).
Các biến thể khác nhau nhưng chung hình dáng: *số → khoảng trắng tuỳ ý → từ chỉ phòng ngủ*.

In [ ]:
vi_du = pd.Series([
    "Modern 1BR Oasis",          # phải khớp: 1
    "Casa 3 dorm centro",        # phải khớp: 3
    "2 bedrooms near park",      # phải khớp: 2
    "Depto frente al metro",     # KHÔNG khớp
    "Habitación año 2024",       # KHÔNG khớp (2024 không phải số phòng!)
])

PAT = r"(\d+)\s*(?:BR|bed|dorm|hab)"
vi_du.str.extract(PAT, expand=False)

Đọc mẫu từng mảnh: `(\d+)` — *lấy* một-hay-nhiều chữ số; `\s*` — khoảng trắng tuỳ ý;
`(?:BR|bed|dorm|hab)` — một trong các từ khoá, nhóm *không lấy*.

Chú ý ca 5: "año **2024**" không bị khớp nhầm vì sau số không có từ khoá phòng ngủ.
**Bộ ví dụ phải chứa cả ca không-được-khớp** — đó là cách kiểm chứng regex.

In [ ]:
# Áp cả cột thật + kiểm đếm
df["so_phong_ngu"] = df["name"].str.extract(PAT, expand=False).astype(float)
print("Số tên tiết lộ số phòng ngủ:", df["so_phong_ngu"].notna().sum())
df.loc[df["so_phong_ngu"].notna(), ["name", "so_phong_ngu"]].head(3)

In [ ]:
# Cùng bài giá tiền — phiên bản regex (chặt hơn 2 lần replace)
gia_regex = (df["price"]
             .str.extract(r"\$([\d,]+)", expand=False)
             .str.replace(",", "")
             .astype(float))
print("Khớp với cách replace:", (gia_regex == df["price_num"]).all())

## 3. Cột chữ → cột tín hiệu

In [ ]:
df["gan_metro"] = gan_metro
df["co_view"] = df["name"].str.contains(r"view|vista", case=False, na=False)
df["co_dau_tbn"] = df["name"].str.contains(r"ción|ñ|á|é|í|ó|ú", case=False, na=False)

df[["gan_metro", "co_view", "co_dau_tbn"]].mean().round(3)

In [ ]:
# Tín hiệu mới trả lời câu hỏi thật: khoe metro thì giá thế nào?
df.groupby("gan_metro")["price_num"].agg(["median", "size"]).round(1)

Phòng khoe "gần metro" *rẻ hơn* — tín hiệu trích đúng, nhưng diễn giải phải cẩn trọng
(phòng nhỏ khu trung tâm hay rao kiểu này). Trích xuất ≠ kết luận — bài của buổi 14.

## 4. Bài tập tại lớp

### Bài 1 — Từ điển tiện nghi

Dùng `tien_nghi` (mục 1): tính **tỷ lệ listing** có (a) máy giặt (`Washer`), (b) điều hoà
(chú ý: có nhiều biến thể "Air conditioning", "AC unit"… — dùng contains), (c) chỗ đỗ xe
(`parking`, cũng nhiều biến thể). Gợi ý: quay về mức listing bằng
`df["amenities"].str.contains(...)` cho nhanh.

In [ ]:
# TODO Bài 1:
for nhan, pat in [("Máy giặt", r"Washer"), ("Điều hoà", r"[Aa]ir condition|AC unit"),
                  ("Đỗ xe", r"parking")]:
    ty_le = df["amenities"].str.contains(pat, case=False, na=False).mean()
    print(f"{nhan:10}: {ty_le:.0%}")

### Bài 2 — Viết regex có kiểm chứng

Nhiều tên phòng ghi diện tích: "Depto 45 m2", "Studio 30m²", "80 M2 luxury". Hãy:

1. Tự viết 5 ví dụ (3 khớp, 2 không — ví dụ "Torre 2024" không được khớp).
2. Viết regex trích **con số diện tích**; chạy trên 5 ví dụ.
3. Áp lên `df["name"]`, đếm số listing khai diện tích, xem 5 dòng khớp đầu tiên.

In [ ]:
# TODO Bài 2 (scaffold — hoàn thiện pattern):
PAT_M2 = r"(\d+)\s*[mM]2?²?\b"
test = pd.Series(["Depto 45 m2", "Studio 30m²", "80 M2 luxury", "Torre 2024", "Casa centro"])
print(test.str.extract(PAT_M2, expand=False).tolist())

df["dien_tich"] = df["name"].str.extract(PAT_M2, expand=False).astype(float)
print("Số listing khai diện tích:", df["dien_tich"].notna().sum())
df.loc[df["dien_tich"].notna(), ["name", "dien_tich"]].head(5)

### Bài 3 — Chuẩn hoá để đếm đúng

`neighbourhood_cleansed` của Santiago khá sạch, nhưng thử làm bẩn rồi cứu lại:
chạy cell dưới tạo cột `quan_ban` (bẩn), rồi viết chuỗi `.str` chuẩn hoá sao cho
`value_counts()` của bản đã cứu **khớp** với bản gốc.

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
bien_the = {0: lambda s: s, 1: lambda s: s.upper(), 2: lambda s: "  " + s + " ",
            3: lambda s: s.lower()}
df["quan_ban"] = [bien_the[k](s) for k, s in
                  zip(rng.integers(0, 4, len(df)), df["neighbourhood_cleansed"])]

# TODO: chuẩn hoá quan_ban về dạng gốc (gợi ý: strip + title — thử và ĐỐI CHIẾU)
df["quan_cuu"] = df["quan_ban"].str.strip().str.title()
goc = df["neighbourhood_cleansed"].str.title().value_counts()
cuu = df["quan_cuu"].value_counts()
print("Khớp hoàn toàn:", goc.equals(cuu))

## 5. Thử thách về nhà 🏆 — Từ điển khía cạnh cho BTL

Hợp phần LLM của bài tập lớn yêu cầu một **baseline regex/từ khoá**. Làm nháp ngay:

1. Chọn 4 khía cạnh của chỗ ở (vị trí, sạch sẽ, chủ nhà, giá trị…).
2. Với mỗi khía cạnh, viết pattern từ khoá **hai ngôn ngữ** (Anh + Tây Ban Nha) — như
   `ASPECT_KEYWORDS` bạn sẽ gặp ở buổi 11.
3. Áp lên cột `name` (tạm thay cho reviews): tỷ lệ nhắc đến từng khía cạnh?
4. Soi 10 dòng khớp mỗi khía cạnh — bao nhiêu là khớp "oan"? Ghi lại 2–3 ca oan thú vị nhất.

Bước 4 chính là "phân tích lỗi baseline" — làm quen trước, buổi 11 đỡ bỡ ngỡ.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    ASPECT_KEYWORDS = {
        "vi_tri": r"location|central|centro|metro|ubicaci",
        # TODO: thêm 3 khía cạnh nữa
    }
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| strip/lower trước khi đếm; `na=False` khi contains | Đếm đúng, lọc không dính NaN |
| explode cho ô đa trị (amenities) | Mở khoá cả một họ câu hỏi mới |
| Regex = hình dáng; `str.extract` + nhóm `( )` | Moi số liệu từ văn bản tự do |
| Kiểm chứng regex bằng bộ ví dụ khớp + không-khớp | Quy trình, không phải may rủi |
| Cột chữ → cột tín hiệu → groupby | Tiền truyện trực tiếp của buổi 11 (LLM) |

**Buổi sau:** dữ liệu thời gian — datetime, resample, và câu hỏi "so với cùng kỳ".